# ContraTICO — Metrics Extension

Runs two evaluation metrics on contraTICO baseline data:
- **LLM Judge** — Qwen2.5-3B-Instruct as NLI judge
- **NLI Classifier** — `facebook/bart-large-mnli`
- **Agreement Rate** — Measures label agreement between both methods

Processes 42 rows × 3 configs × 8 perturbations × 5 languages = ~5,040 total.

## 0. Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
elif IN_KAGGLE:
    print('Running on Kaggle')
else:
    print('Running locally')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'sentencepiece'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## 1. Pre-download Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
import torch

print('=== Downloading/Loading Models ===')

# Qwen (for LLM Judge)
QWEN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
print(f'[1/2] Loading {QWEN_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ Qwen cached')

# BART-MNLI (for NLI Classifier)
NLI_MODEL = 'facebook/bart-large-mnli'
print(f'[2/2] Loading {NLI_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
del model, tokenizer
print('      ✓ BART-MNLI cached')

print('\n=== All models cached! ===')

## 2. Path Configuration

In [ ]:
EXTENSION_DIR = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/contratico/metrics-extension"
CODE_DIR = f"{EXTENSION_DIR}/code"
BASELINE_DIR = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/contratico/baseline"

CONFIGS = ['vanilla', 'atomic', 'semantic']
LANGUAGES = ['es', 'fr', 'hi', 'tl', 'zh']
MAX_ROWS = 42

# Verify baseline exists
for config in CONFIGS:
    source_path = os.path.join(BASELINE_DIR, 'QA', 'source', f'en-{config}.jsonl')
    if os.path.exists(source_path):
        with open(source_path) as f:
            n = sum(1 for _ in f)
        print(f'✓ {config} source: {n} rows')
    else:
        print(f'✗ {config} source: NOT FOUND')

print(f'\nWill process {MAX_ROWS} rows per file')
print(f'Total estimated: {MAX_ROWS} × 3 configs × 8 perts × 5 langs = {MAX_ROWS * 3 * 8 * 5}')

## 3. LLM Judge Evaluation

Run LLM Judge on all 3 configs.

In [ ]:
for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"LLM Judge - Config: {config}")
    print(f"{'='*60}")

    cmd = [
        sys.executable, '-u',
        f'{CODE_DIR}/llm_judge_contratico.py',
        '--config', config,
        '--baseline_dir', BASELINE_DIR,
        '--output_dir', EXTENSION_DIR,
        '--max_rows', str(MAX_ROWS),
    ]

    subprocess.run(cmd, check=True)
    print(f'✓ LLM Judge {config} complete!')

print('\n✓ All LLM Judge evaluations complete!')

## 4. NLI Classifier Evaluation

Run NLI Classifier on all 3 configs.

In [ ]:
for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"NLI Classifier - Config: {config}")
    print(f"{'='*60}")

    cmd = [
        sys.executable, '-u',
        f'{CODE_DIR}/nli_classifier_contratico.py',
        '--config', config,
        '--baseline_dir', BASELINE_DIR,
        '--output_dir', EXTENSION_DIR,
        '--max_rows', str(MAX_ROWS),
    ]

    subprocess.run(cmd, check=True)
    print(f'✓ NLI Classifier {config} complete!')

print('\n✓ All NLI Classifier evaluations complete!')

## 5. Agreement Rate

Computes the agreement between NLI Classifier and LLM Judge labels.
For each answer pair, both methods produce a label (entailment / neutral / contradiction).
Agreement Rate = fraction of samples where both methods agree.

In [ ]:
import json
import os

def load_labels(filepath):
    """Load labels from a JSONL results file."""
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line.strip())
            label = data.get('label', data.get('nli_label', data.get('llm_label', '')))
            labels.append(label.lower().strip())
    return labels

print(f"{'='*70}")
print('Agreement Rate: NLI Classifier vs LLM Judge')
print(f"{'='*70}")

grand_agree = 0
grand_total = 0

for config in CONFIGS:
    print(f"\n  Config: {config}")
    print(f"  {'-'*40}")

    config_agree = 0
    config_total = 0

    for lang in LANGUAGES:
        nli_path = os.path.join(EXTENSION_DIR, 'results', 'nli', config, f'{lang}-nli.jsonl')
        llm_path = os.path.join(EXTENSION_DIR, 'results', 'llm-judge', config, f'{lang}-llm-judge.jsonl')

        if not os.path.exists(nli_path) or not os.path.exists(llm_path):
            print(f'    {lang}: ⚠ Missing files')
            continue

        nli_labels = load_labels(nli_path)
        llm_labels = load_labels(llm_path)

        n = min(len(nli_labels), len(llm_labels))
        if n == 0:
            print(f'    {lang}: ⚠ No labels')
            continue

        agree = sum(1 for i in range(n) if nli_labels[i] == llm_labels[i])
        rate = agree / n * 100

        config_agree += agree
        config_total += n

        print(f'    {lang:>5}: {agree}/{n} = {rate:.1f}%')

    if config_total > 0:
        config_rate = config_agree / config_total * 100
        print(f'    {"": >5}  ──────────')
        print(f'    {"Total":>5}: {config_agree}/{config_total} = {config_rate:.1f}%')

    grand_agree += config_agree
    grand_total += config_total

if grand_total > 0:
    overall = grand_agree / grand_total * 100
    print(f"\n{'='*70}")
    print(f'  OVERALL Agreement: {grand_agree}/{grand_total} = {overall:.1f}%')
    print(f"{'='*70}")
else:
    print('\n  ⚠ No results to compare — run Steps 3 and 4 first')

## 6. Verification

In [ ]:
import json

for eval_type in ['llm-judge', 'nli']:
    print(f"\n{'='*40}")
    print(f"{eval_type.upper()} Results")
    print(f"{'='*40}")
    for config in CONFIGS:
        total = 0
        for lang in LANGUAGES:
            suffix = 'llm-judge' if eval_type == 'llm-judge' else 'nli'
            path = os.path.join(EXTENSION_DIR, 'results', eval_type, config, f'{lang}-{suffix}.jsonl')
            if os.path.exists(path):
                with open(path) as f:
                    n = sum(1 for _ in f)
                total += n
            else:
                print(f'  ✗ {config}/{lang}: NOT FOUND')
        print(f'  {config}: {total} total rows across {len(LANGUAGES)} languages')

print('\n✓ Verification complete!')

## Summary

Metrics Extension complete! Results saved to:
- **LLM Judge**: `results/llm-judge/{config}/{lang}-llm-judge.jsonl`
- **NLI Classifier**: `results/nli/{config}/{lang}-nli.jsonl`
- **Agreement Rate**: printed in Step 5 (NLI vs LLM Judge label concordance per config × language)